# C4 — Boosting

Avalia `HistGradientBoostingClassifier` na mesma partição do C3.
Se `G6_finalistas_tuning` estiver aprovado, executa busca limitada para
ambos os finalistas. Não abre o teste, não calibra e não escolhe limiar.

**Pré-requisito:** `pip install -e .`

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact, split_partitions
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas
from src.models.logistic import build_logistic_pipeline
from src.models.boosting import build_boosting_pipeline

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(C.SEED)
print(C.resumo())

## Carregar artefato e materializar partições

`split_partitions` devolve apenas `(treino, validacao)` — o teste fica inacessível.

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=EXPECTED_KIND,
    partitions=("treino", "validacao"),
)
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES].copy()
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

check_leakage(X_train)
check_leakage(X_val)

n_teste = int((df[PARTICAO].astype(str) == "teste").sum())

display(pd.DataFrame({
    "particao":    ["treino", "validacao"],
    "n":           [len(treino), len(validacao)],
    "positivos":   [int(y_train.sum()), int(y_val.sum())],
    "prevalencia": [float(y_train.mean()), float(y_val.mean())],
}))

print(f"cohort_kind : {manifest['cohort_kind']}")
print(f"features    : {len(FEATURES)}")
print(f"teste       : {n_teste:,} (fechado)")

## Modelos de referência

Logística e boosting com configurações do YAML. Servem de base para a
comparação caso o gate G6 esteja pendente.

In [ ]:
log_cfg   = C.cfg("models", "logistic") or {}
boost_cfg = C.cfg("models", "boosting") or {}

DEFAULTS_BOOST = {
    "max_leaf_nodes":    int(boost_cfg.get("max_leaf_nodes") or 31),
    "min_samples_leaf":  int(boost_cfg.get("min_samples_leaf") or 20),
    "l2_regularization": float(boost_cfg.get("l2_regularization") or 0.0),
    "early_stopping":    bool(boost_cfg.get("early_stopping") or False),
    "class_weight":      boost_cfg.get("class_weight") or "balanced",
}

DEFAULTS_LOG = {
    "solver":       log_cfg.get("solver", "lbfgs"),
    "max_iter":     int(log_cfg.get("max_iter", 1000)),
    "class_weight": log_cfg.get("class_weight", "balanced"),
}

pipe_log_ref = build_logistic_pipeline(C=1.0, **DEFAULTS_LOG)
pipe_log_ref.fit(X_train, y_train)
prob_log_ref = pipe_log_ref.predict_proba(X_val)[:, 1]

pipe_boost_ref = build_boosting_pipeline(
    learning_rate=0.05, max_iter=500, max_depth=None, **DEFAULTS_BOOST,
)
pipe_boost_ref.fit(X_train, y_train)
prob_boost_ref = pipe_boost_ref.predict_proba(X_val)[:, 1]

display(pd.DataFrame([
    {"modelo": "logistica_C1",  **metricas_completas(y_val, prob_log_ref,   limiar=0.5)},
    {"modelo": "boosting_base", **metricas_completas(y_val, prob_boost_ref, limiar=0.5)},
]))

## Busca limitada (somente se G6 aprovado)

Respeita `tuning.metrica_principal` e `tuning.max_configuracoes_por_modelo`.
Amostra apenas se `MAX_TREINO_BUSCA` estiver definido.

In [ ]:
MAX_TREINO_BUSCA = None

METRICAS = ("ap", "pr_auc", "auc", "brier", "f1", "precision", "recall")


def _subamostrar(X, y, max_n, seed):
    if max_n is None or max_n <= 0 or len(X) <= max_n:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=max_n, replace=False)
    return X.iloc[idx].copy(), y.iloc[idx].copy()


def _busca_limitada(candidatos, construir, X_tr, y_tr, X_va, y_va, *,
                    seed, max_configs, metrica_principal):
    rng = np.random.default_rng(seed)
    ordem = rng.permutation(len(candidatos))
    selecionados = [candidatos[i] for i in ordem][:max_configs]

    linhas, melhor = [], None
    for params in selecionados:
        t0 = time.perf_counter()
        pipe = construir(**params)
        pipe.fit(X_tr, y_tr)
        fit_s = time.perf_counter() - t0
        prob = pipe.predict_proba(X_va)[:, 1]
        m = metricas_completas(y_va, prob, limiar=0.5)

        linha = {**{k: float(m[k]) for k in METRICAS},
                 **params,
                 "duracao_fit_s": float(fit_s)}
        linhas.append(linha)

        score = linha[metrica_principal]
        if melhor is None or score > melhor["score"]:
            melhor = {"score": score, "params": params,
                      "pipeline": pipe, "prob_val": prob}

    return pd.DataFrame(linhas), melhor


def _candidatos_boosting():
    return [
        {"learning_rate": float(lr), "max_iter": int(mi),
         "max_depth": (None if md is None else int(md))}
        for lr in (C.cfg("models", "boosting", "busca_learning_rate") or [0.05])
        for mi in (C.cfg("models", "boosting", "busca_max_iter")      or [500])
        for md in (C.cfg("models", "boosting", "busca_max_depth")     or [None])
    ]


def _candidatos_logistica():
    return [{"C": float(c)} for c in (C.cfg("models", "logistic", "busca_C") or [1.0])]

In [ ]:
G6_STATUS  = (C.cfg("gates", "G6_finalistas_tuning") or {}).get("status")
G6_APROVADO = G6_STATUS == "aprovado"

tuning_cfg        = C.cfg("tuning") or {}
METRICA_PRINCIPAL = str(tuning_cfg.get("metrica_principal") or "pr_auc")
MAX_CONFIGS       = max(1, int(tuning_cfg.get("max_configuracoes_por_modelo") or 30))

print(f"G6_finalistas_tuning: {G6_STATUS} (aprovado={G6_APROVADO})")
print(f"metrica_principal   : {METRICA_PRINCIPAL}")
print(f"max_configs         : {MAX_CONFIGS}")

resultados_busca_log = resultados_busca_boost = None
melhor_config_log    = melhor_config_boost    = None

pipe_log_final,   prob_log_final   = pipe_log_ref,   prob_log_ref
pipe_boost_final, prob_boost_final = pipe_boost_ref, prob_boost_ref
metricas_log_final   = {**metricas_completas(y_val, prob_log_final,   limiar=0.5), "modelo": "logistica_C1"}
metricas_boost_final = {**metricas_completas(y_val, prob_boost_final, limiar=0.5), "modelo": "boosting_base"}

if G6_APROVADO:
    X_busca, y_busca = _subamostrar(X_train, y_train, MAX_TREINO_BUSCA, SEED)
    print(f"\nBusca em {len(X_busca):,} linhas (treino original: {len(X_train):,}).")

    resultados_busca_log, melhor_config_log = _busca_limitada(
        _candidatos_logistica(),
        lambda **p: build_logistic_pipeline(**p, **DEFAULTS_LOG),
        X_busca, y_busca, X_val, y_val,
        seed=SEED, max_configs=MAX_CONFIGS, metrica_principal=METRICA_PRINCIPAL,
    )
    resultados_busca_log = resultados_busca_log.sort_values(
        [METRICA_PRINCIPAL, "brier"], ascending=[False, True]
    ).reset_index(drop=True)
    display(resultados_busca_log)

    resultados_busca_boost, melhor_config_boost = _busca_limitada(
        _candidatos_boosting(),
        lambda **p: build_boosting_pipeline(**p, **DEFAULTS_BOOST),
        X_busca, y_busca, X_val, y_val,
        seed=SEED, max_configs=MAX_CONFIGS, metrica_principal=METRICA_PRINCIPAL,
    )
    resultados_busca_boost = resultados_busca_boost.sort_values(
        [METRICA_PRINCIPAL, "brier"], ascending=[False, True]
    ).reset_index(drop=True)
    display(resultados_busca_boost)

    if melhor_config_log is not None:
        pipe_log_final = build_logistic_pipeline(**melhor_config_log["params"], **DEFAULTS_LOG)
        pipe_log_final.fit(X_train, y_train)
        prob_log_final = pipe_log_final.predict_proba(X_val)[:, 1]
        metricas_log_final = {**metricas_completas(y_val, prob_log_final, limiar=0.5),
                              "modelo": "logistica_tuned"}

    if melhor_config_boost is not None:
        pipe_boost_final = build_boosting_pipeline(**melhor_config_boost["params"], **DEFAULTS_BOOST)
        pipe_boost_final.fit(X_train, y_train)
        prob_boost_final = pipe_boost_final.predict_proba(X_val)[:, 1]
        metricas_boost_final = {**metricas_completas(y_val, prob_boost_final, limiar=0.5),
                                "modelo": "boosting_tuned"}
else:
    print("\nG6 pendente — busca limitada não executada. Comparação usará referências.")

## Comparação, predições e persistência

In [ ]:
comparacao = pd.DataFrame([
    {"modelo": metricas_log_final["modelo"],
     **{k: metricas_log_final[k] for k in ("ap", "pr_auc", "auc", "brier", "f1", "precision", "recall", "accuracy")}},
    {"modelo": metricas_boost_final["modelo"],
     **{k: metricas_boost_final[k] for k in ("ap", "pr_auc", "auc", "brier", "f1", "precision", "recall", "accuracy")}},
]).sort_values("ap", ascending=False).reset_index(drop=True)

delta_ap = (
    comparacao.loc[comparacao["modelo"].str.startswith("boosting"),  "ap"].iloc[0]
    - comparacao.loc[comparacao["modelo"].str.startswith("logistica"), "ap"].iloc[0]
)
display(comparacao)
print(f"Δ AP (boosting − logística): {delta_ap:+.4f}")

pred_val = pd.concat([
    pd.DataFrame({"y_true": y_val.to_numpy(), "y_prob": prob_log_final,
                  "modelo": metricas_log_final["modelo"],
                  "cohort_kind": manifest["cohort_kind"]}),
    pd.DataFrame({"y_true": y_val.to_numpy(), "y_prob": prob_boost_final,
                  "modelo": metricas_boost_final["modelo"],
                  "cohort_kind": manifest["cohort_kind"]}),
], ignore_index=True)
pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"C4_boosting_{manifest['cohort_kind']}_{timestamp}"

caminho_comparacao = TABLES_DIR      / f"{prefixo}_comparacao.csv"
caminho_predicoes  = PREDICTIONS_DIR / f"{prefixo}_predicoes_validacao.parquet"
caminho_manifesto  = TABLES_DIR      / f"{prefixo}_manifesto.json"
caminho_busca_log  = TABLES_DIR      / f"{prefixo}_busca_logistica.csv"
caminho_busca_bst  = TABLES_DIR      / f"{prefixo}_busca_boosting.csv"

comparacao.to_csv(caminho_comparacao, index=False, encoding="utf-8-sig")
pred_val.to_parquet(caminho_predicoes, index=False)
if resultados_busca_log   is not None: resultados_busca_log.to_csv(caminho_busca_log, index=False, encoding="utf-8-sig")
if resultados_busca_boost is not None: resultados_busca_boost.to_csv(caminho_busca_bst, index=False, encoding="utf-8-sig")

manifesto_c4 = {
    "cohort_kind":        manifest["cohort_kind"],
    "dataset_path":       manifest["dataset_path"],
    "dataset_sha256":     manifest["dataset_sha256"],
    "feature_columns":    FEATURES,
    "target_column":      ALVO,
    "partition_column":   PARTICAO,
    "n_treino":           int(len(treino)),
    "n_validacao":        int(len(validacao)),
    "n_teste_fechado":    n_teste,
    "g6_status":          G6_STATUS,
    "buscas_executadas":  bool(G6_APROVADO),
    "metrica_principal":  METRICA_PRINCIPAL,
    "max_configs":        MAX_CONFIGS,
    "melhor_config_logistica": (
        None if melhor_config_log is None
        else {"C": melhor_config_log["params"]["C"],
              "score": float(melhor_config_log["score"])}
    ),
    "melhor_config_boosting": (
        None if melhor_config_boost is None
        else {**melhor_config_boost["params"],
              "score": float(melhor_config_boost["score"])}
    ),
    "limiar_usado":        0.5,
    "limiar_e_diagnostico": True,
    "calibracao_aplicada": False,
    "teste_aberto":        False,
    "features_final_congelada_no_yaml": bool(C.cfg("features", "final")),
    "artefatos": {
        "comparacao":          str(caminho_comparacao),
        "predicoes_validacao": str(caminho_predicoes),
        "busca_logistica":     str(caminho_busca_log) if resultados_busca_log   is not None else None,
        "busca_boosting":      str(caminho_busca_bst) if resultados_busca_boost is not None else None,
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c4, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

assert len(pd.read_csv(caminho_comparacao, encoding="utf-8-sig")) == len(comparacao)
assert len(pd.read_parquet(caminho_predicoes)) == len(pred_val)

print(f"Comparação : {caminho_comparacao}")
print(f"Predições  : {caminho_predicoes}")
print(f"Manifesto  : {caminho_manifesto}")

## Proposta de atualização do YAML

Consolida os hiperparâmetros vencedores num bloco pronto para colar em
`configs/supervised.yaml`. **Não é aplicada automaticamente** — o C5 lê o
YAML que existir em disco.

In [ ]:
def _fmt(valor):
    if valor is None:
        return "null"
    if isinstance(valor, bool):
        return "true" if valor else "false"
    if isinstance(valor, str):
        return repr(valor)
    return str(valor)


def _montar_proposta_yaml():
    destino = C.SUPERVISED_YAML.relative_to(C.ROOT_DIR).as_posix()
    linhas = [
        "# Proposta de atualização — gerada pelo C4.",
        f"# Destino: {destino}",
        f"# NÃO aplicada automaticamente; revise e cole em {destino}.",
        f"# cohort_kind: {manifest['cohort_kind']}",
        f"# g6_status: {G6_STATUS}",
        f"# metrica_principal: {METRICA_PRINCIPAL}",
        "models:",
    ]
    if melhor_config_log is not None:
        linhas += ["  logistic:", f"    C: {_fmt(melhor_config_log['params']['C'])}"]
    if melhor_config_boost is not None:
        p = melhor_config_boost["params"]
        linhas += [
            "  boosting:",
            f"    learning_rate: {_fmt(p['learning_rate'])}",
            f"    max_iter: {_fmt(p['max_iter'])}",
            f"    max_depth: {_fmt(p['max_depth'])}",
        ]
    return "\n".join(linhas) + "\n"


if G6_APROVADO and (melhor_config_log or melhor_config_boost):
    proposta = _montar_proposta_yaml()
    caminho_proposta = TABLES_DIR / f"{prefixo}_proposta_yaml.yaml"
    caminho_proposta.write_text(proposta, encoding="utf-8")
    print(proposta)
    print(f"Salvo: {caminho_proposta}")
else:
    print("Sem proposta de atualização do YAML (G6 pendente ou busca vazia).")

In [ ]:
with Run(
    modelo="C4_boosting",
    parametros={
        "cohort_kind":         manifest["cohort_kind"],
        "g6_status":           G6_STATUS,
        "buscas_executadas":   bool(G6_APROVADO),
        "metrica_principal":   METRICA_PRINCIPAL,
        "max_configs":         MAX_CONFIGS,
        "limiar_diagnostico":  0.5,
        "calibracao_aplicada": False,
        "teste_aberto":        False,
    },
    dataset={
        "n_treino":        int(len(treino)),
        "n_validacao":     int(len(validacao)),
        "n_teste_fechado": n_teste,
    },
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas="C4 — boosting na partição do C3. Buscas condicionadas ao G6. Teste não consultado.",
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(
        logistica_ap=metricas_log_final["ap"],
        logistica_auc=metricas_log_final["auc"],
        boosting_ap=metricas_boost_final["ap"],
        boosting_auc=metricas_boost_final["auc"],
        boosting_brier=metricas_boost_final["brier"],
        delta_ap=delta_ap,
    )

print("Run registrado em:", r.pasta)